# Evaluation Techniques for Logistic Regression

## Overview
This notebook provides a comprehensive guide to evaluating logistic regression models. We'll explore various metrics and techniques to assess model performance, understand their strengths and weaknesses, and learn how to interpret them in different contexts.

## Key Evaluation Topics
1. Confusion Matrix and Basic Metrics (Accuracy, Precision, Recall, F1-Score)
2. Receiver Operating Characteristic (ROC) Curve and Area Under Curve (AUC)
3. Precision-Recall Curve
4. Cross-Validation Techniques
5. Model Calibration and Probability Assessment
6. Evaluation for Imbalanced Datasets

Let's dive into these evaluation techniques with practical examples.

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import roc_curve, auc, precision_recall_curve
from sklearn.calibration import calibration_curve
from imblearn.over_sampling import SMOTE

# Set random seed for reproducibility
np.random.seed(42)

## 1. Confusion Matrix and Basic Metrics

The confusion matrix is a fundamental tool for evaluating binary classification models. It shows the number of correct and incorrect predictions broken down by each class.

From the confusion matrix, we can derive several key metrics:
- **Accuracy**: (TP + TN) / (TP + TN + FP + FN)
- **Precision**: TP / (TP + FP)
- **Recall (Sensitivity)**: TP / (TP + FN)
- **F1-Score**: 2 * (Precision * Recall) / (Precision + Recall)

Where:
- TP = True Positives
- TN = True Negatives
- FP = False Positives
- FN = False Negatives

In [ ]:
# Generate synthetic dataset
X, y = make_classification(n_samples=1000, n_features=10, random_state=42)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train model
model = LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)

# Make predictions
y_pred = model.predict(X_test_scaled)

# Calculate confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

# Calculate and print basic metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f'Accuracy: {accuracy:.3f}')
print(f'Precision: {precision:.3f}')
print(f'Recall: {recall:.3f}')
print(f'F1-Score: {f1:.3f}')

# Detailed classification report
print('
Detailed Classification Report:')
print(classification_report(y_test, y_pred))

### Interpreting Basic Metrics
- **Accuracy** is useful when classes are balanced, but can be misleading for imbalanced datasets.
- **Precision** is important when the cost of false positives is high (e.g., spam detection).
- **Recall** is crucial when the cost of false negatives is high (e.g., disease detection).
- **F1-Score** provides a balance between precision and recall, useful when both false positives and false negatives are important.

## 2. Receiver Operating Characteristic (ROC) Curve and AUC

The ROC curve plots the True Positive Rate (Recall) against the False Positive Rate at various threshold settings. The Area Under the Curve (AUC) provides a single number summarizing the model's ability to discriminate between classes.

- AUC = 0.5: No discrimination (random guessing)
- AUC = 1.0: Perfect discrimination
- AUC < 0.5: Worse than random guessing

In [ ]:
# Get predicted probabilities
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

# Calculate ROC curve and AUC
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)

# Plot ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.show()

print(f'AUC Score: {roc_auc:.3f}')

### Interpreting ROC and AUC
- The ROC curve shows the trade-off between sensitivity (recall) and specificity (1 - FPR).
- AUC is useful for comparing models - higher AUC generally indicates better performance.
- However, AUC can be misleading for highly imbalanced datasets as it may be high even when the model struggles to identify the minority class.

## 3. Precision-Recall Curve

The Precision-Recall curve plots precision against recall at various threshold settings. It's particularly useful for imbalanced datasets where the positive class is of more interest.

In [ ]:
# Calculate precision-recall curve
precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba)

# Plot precision-recall curve
plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color='darkorange', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.show()

# Calculate average precision score
from sklearn.metrics import average_precision_score
ap_score = average_precision_score(y_test, y_pred_proba)
print(f'Average Precision Score: {ap_score:.3f}')

### Interpreting Precision-Recall Curve
- Unlike ROC, the PR curve focuses on the positive class performance.
- A good model will maintain high precision as recall increases (curve stays high).
- The Average Precision score summarizes the PR curve into a single number - higher is better.
- This is more informative than AUC-ROC for imbalanced datasets.

## 4. Cross-Validation Techniques

Cross-validation provides a more robust estimate of model performance by splitting the data into multiple train-test sets. Common approaches include k-fold cross-validation and stratified k-fold for imbalanced data.

In [ ]:
# Perform 5-fold cross-validation for multiple metrics
metrics = {'accuracy': [], 'precision': [], 'recall': [], 'f1': []}
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for train_idx, test_idx in kf.split(X):
    X_train_cv, X_test_cv = X[train_idx], X[test_idx]
    y_train_cv, y_test_cv = y[train_idx], y[test_idx]
    
    # Scale features
    scaler = StandardScaler()
    X_train_cv_scaled = scaler.fit_transform(X_train_cv)
    X_test_cv_scaled = scaler.transform(X_test_cv)
    
    # Train model
    model_cv = LogisticRegression(random_state=42)
    model_cv.fit(X_train_cv_scaled, y_train_cv)
    
    # Make predictions
    y_pred_cv = model_cv.predict(X_test_cv_scaled)
    
    # Calculate metrics
    metrics['accuracy'].append(accuracy_score(y_test_cv, y_pred_cv))
    metrics['precision'].append(precision_score(y_test_cv, y_pred_cv))
    metrics['recall'].append(recall_score(y_test_cv, y_pred_cv))
    metrics['f1'].append(f1_score(y_test_cv, y_pred_cv))

# Plot cross-validation results
plt.figure(figsize=(10, 6))
plt.boxplot(metrics.values(), labels=metrics.keys())
plt.title('5-Fold Cross-Validation Results')
plt.ylabel('Score')
plt.show()

# Print mean and standard deviation for each metric
for metric, scores in metrics.items():
    print(f'{metric.capitalize()} - Mean: {np.mean(scores):.3f}, Std: {np.std(scores):.3f}')

### Interpreting Cross-Validation Results
- The mean score across folds gives a better estimate of model performance on unseen data.
- The standard deviation indicates model stability - high variation suggests the model is sensitive to the training data split.
- For imbalanced datasets, use StratifiedKFold to maintain class proportions in each fold.

## 5. Model Calibration and Probability Assessment

Logistic regression outputs probabilities that should ideally reflect the true likelihood of the positive class. Calibration curves help assess how well-calibrated these probabilities are.

In [ ]:
# Plot calibration curve
prob_true, prob_pred = calibration_curve(y_test, y_pred_proba, n_bins=10)

plt.figure(figsize=(8, 6))
plt.plot(prob_pred, prob_true, marker='o', label='Logistic Regression')
plt.plot([0, 1], [0, 1], linestyle='--', label='Perfectly Calibrated')
plt.xlabel('Mean Predicted Probability')
plt.ylabel('Fraction of Positives')
plt.title('Calibration Curve')
plt.legend()
plt.show()

# Brier score - lower is better
from sklearn.metrics import brier_score_loss
brier_score = brier_score_loss(y_test, y_pred_proba)
print(f'Brier Score: {brier_score:.3f}')

### Interpreting Calibration Results
- Points close to the diagonal line indicate well-calibrated probabilities.
- Points above the diagonal mean the model underestimates probabilities; below means it overestimates.
- Brier score measures the mean squared difference between predicted probabilities and actual outcomes - lower scores indicate better calibration.
- Poorly calibrated models can be improved with techniques like Platt scaling or isotonic regression.

## 6. Evaluation for Imbalanced Datasets

When dealing with imbalanced datasets, standard metrics like accuracy can be misleading. We need to focus on metrics that emphasize the minority class performance.

In [ ]:
# Generate imbalanced dataset
X_imb, y_imb = make_classification(n_samples=1000, n_features=10, weights=[0.9, 0.1], random_state=42)

# Split the data
X_train_imb, X_test_imb, y_train_imb, y_test_imb = train_test_split(
    X_imb, y_imb, test_size=0.2, random_state=42
)

# Scale features
scaler = StandardScaler()
X_train_imb_scaled = scaler.fit_transform(X_train_imb)
X_test_imb_scaled = scaler.transform(X_test_imb)

# Train model without balancing
model_imb = LogisticRegression(random_state=42)
model_imb.fit(X_train_imb_scaled, y_train_imb)

# Evaluate without balancing
y_pred_imb = model_imb.predict(X_test_imb_scaled)
y_pred_proba_imb = model_imb.predict_proba(X_test_imb_scaled)[:, 1]

print('Results without handling imbalance:')
print(classification_report(y_test_imb, y_pred_imb))

# Now train with balancing using SMOTE
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_imb_scaled, y_train_imb)

model_balanced = LogisticRegression(random_state=42)
model_balanced.fit(X_train_balanced, y_train_balanced)

# Evaluate with balancing
y_pred_balanced = model_balanced.predict(X_test_imb_scaled)
y_pred_proba_balanced = model_balanced.predict_proba(X_test_imb_scaled)[:, 1]

print('
Results after handling imbalance with SMOTE:')
print(classification_report(y_test_imb, y_pred_balanced))

# Compare precision-recall curves
precision_imb, recall_imb, _ = precision_recall_curve(y_test_imb, y_pred_proba_imb)
precision_bal, recall_bal, _ = precision_recall_curve(y_test_imb, y_pred_proba_balanced)

plt.figure(figsize=(8, 6))
plt.plot(recall_imb, precision_imb, color='red', lw=2, label='Without Balancing')
plt.plot(recall_bal, precision_bal, color='green', lw=2, label='With SMOTE Balancing')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve: Imbalanced Data Handling')
plt.legend()
plt.show()

### Interpreting Results for Imbalanced Data
- Notice how accuracy might be high without balancing, but recall for the minority class is often poor.
- After balancing with SMOTE, recall for the minority class typically improves, though overall accuracy might decrease.
- Focus on precision, recall, and F1-score for the minority class rather than overall accuracy.
- Precision-Recall curves are more informative than ROC curves for imbalanced data.
- Other techniques for imbalanced data include class-weight adjustment in the model, different sampling methods, or using anomaly detection approaches.

## Summary of Evaluation Techniques

Choosing the right evaluation metric depends on your specific problem and business context:

- **Balanced datasets**: Accuracy, ROC-AUC, and F1-score are good overall metrics.
- **Imbalanced datasets**: Focus on precision, recall, F1-score for the minority class, and Precision-Recall curves.
- **Cost-sensitive problems**: Consider the relative costs of false positives vs. false negatives and choose metrics accordingly (e.g., prioritize recall for medical diagnosis).
- **Probability interpretation**: Use calibration curves and Brier score if predicted probabilities are important for decision-making.
- **Robust performance estimation**: Always use cross-validation to get a reliable estimate of model performance.

Remember that no single metric tells the whole story - use a combination of these techniques to get a complete picture of your model's performance.